In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, re, ast, json, random, argparse, sys, hashlib
from typing import List, Dict, Any, Tuple
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments, EarlyStoppingCallback
)
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score

# ============================ CONFIG (defaults) ============================

DEF_CFG = {
    # Sentence-level split CSVs (disjoint by seed; no leakage)
    "train_csv": "data/splits/train_seed42.csv",
    "val_csv":   "data/splits/val_seed42.csv",
    "test_csv":  "data/splits/test_seed42.csv",

    # Sentence-level columns
    "TEXT_COL":     "covered_text",
    "LABEL_COL":    "GoldFaceAct",
    "EMAIL_COL":    "email_id",
    "SENTIDX_COL":  "sentence_idx",
    "ISREQ_COL":    "is_request",   # 1 = Request, 0 = Reply
    "SEED_COL":     "seed",
    "PAIR_COL":     "pair_idx",

    # 9 FA labels (order fixed)
    "LABELS": ["HNeg+","HNeg-","HPos+","HPos-","Neutral","SNeg+","SNeg-","SPos+","SPos-"],

    # FA model (BERT + ASL) — Request+Reply sequence-labeling
    "fa_model_name": "bert-base-uncased",
    "max_length": 512,
    "fa_epochs": 5,
    "fa_batch_size": 4,
    "fa_lr": 2e-5,
    "fa_weight_decay": 0.01,
    "fa_early_stop": 2,
    "asl_gamma_pos": 0.0,
    "asl_gamma_neg": 4.0,
    "asl_clip": 0.05,
    "fa_force_tau_f1": 0.60,  # report τ on VAL for comparability

    # default temp dir (validated; may fall back to ~/.cache/opr/)
    "fa_tmpdir": "./fa_models_req_plus_rep",

    # Doc table (targets)
    "DOC_CSV":  "data/corpus/email_text_gold_three_dimensions_politeness_score_with_seed_correct.csv",
    "DOC_ID_COL":   "email_id",
    "DOC_TEXT_COL": "text_email",
    "TARGETS": [
        "Directness_vs_Indirectness__GOLD",
        "Structural_Politeness_and_Politeness_Markers__GOLD",
        "Tone_and_Overall_Consideration__GOLD"
    ],

    # BERT doc-level search (lean defaults)
    "bert_encoder": "bert-base-uncased",
    "bert_lr_grid": [2e-5],
    "bert_epochs_grid": [3, 5],
    "bert_dropout_grid": [0.1],
    "bert_batch_grid": [8],
    "chunk_len_grid": [400],
    "chunk_stride_grid": [350],

    # PredFA-only MLP grid
    "mlp_hidden_grid": [128],
    "mlp_dropout_grid": [0.1],
    "mlp_lr_grid": [1e-3],
    "mlp_epochs": 30,
    "mlp_batch": 256,

    # General
    "output_dir": "./end2end_outputs_req_plus_rep_hspt13",
    "seed": 42,

    # Document-level early stopping (dev score = rho_macro - lambda * mean_MAE)
    "doc_early_stop_patience": 2,
    "doc_early_stop_min_delta": 1e-4,
    "doc_early_stop_lambda": 1e-3,   # small MAE regularizer weight

    # Runtime
    "num_workers": 4,
}

# ============================ Helpers & utils ============================

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def ensure_dir(d): os.makedirs(d, exist_ok=True)
def _safe_device(): return "cuda" if torch.cuda.is_available() else "cpu"
def _expand(p: str) -> str: return os.path.abspath(os.path.expanduser(p))
def _assert_file(p: str): assert os.path.isfile(p), f"Missing file: {p}"

def _ensure_writable_dir(p: str) -> str:
    p = _expand(p)
    try:
        os.makedirs(p, exist_ok=True)
        if os.access(p, os.W_OK):
            return p
    except Exception:
        pass
    fallback_root = _expand("~/.cache/opr")
    fallback = os.path.join(fallback_root, os.path.basename(p.rstrip(os.sep)) or "out")
    os.makedirs(fallback, exist_ok=True)
    if not os.access(fallback, os.W_OK):
        raise PermissionError(f"Cannot write to '{p}' and fallback '{fallback}' is not writable.")
    print(f"[WARN] '{p}' not writable. Using fallback '{fallback}'.")
    return fallback

def parse_labels(cell, LABELS):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)): return []
    s = str(cell).strip()
    if s.startswith('[') and s.endswith(']'):
        try:
            arr = ast.literal_eval(s); return [str(x).strip() for x in arr]
        except Exception:
            pass
    return [t.strip() for t in re.split(r'[;,]', s) if t.strip()]

def to_multi_hot(names: List[str], label2id: Dict[str,int]) -> np.ndarray:
    v = np.zeros(len(label2id), dtype=np.float32)
    for n in names:
        if n in label2id: v[label2id[n]] = 1.0
    return v

def mae_rmse(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred), axis=0)
    rmse = np.sqrt(np.mean((y_true - y_pred)**2, axis=0))
    return mae, rmse

def spearman_each(y_true, y_pred):
    T = y_true.shape[1]
    rhos = []
    for t in range(T):
        rho, _ = spearmanr(y_true[:, t], y_pred[:, t])
        rhos.append(float(0.0 if np.isnan(rho) else rho))
    return rhos

def spearman_macro(y_true, y_pred):
    rhos = spearman_each(y_true, y_pred)
    return float(np.mean(rhos)), rhos

def _scrub(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a, dtype=np.float32)
    a[~np.isfinite(a)] = 0.0
    return a

def sigmoid_stable_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    np.clip(x, -50.0, 50.0, out=x)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)

# ============================ Load & pair building (Req+Rep) ============================

def load_sent_split(csv_path: str, CFG):
    df = pd.read_csv(csv_path)
    need = [CFG["TEXT_COL"], CFG["LABEL_COL"], CFG["EMAIL_COL"], CFG["SENTIDX_COL"],
            CFG["ISREQ_COL"], CFG["SEED_COL"], CFG["PAIR_COL"]]
    for c in need:
        assert c in df.columns, f"Missing '{c}' in {csv_path}"
    for c in [CFG["EMAIL_COL"], CFG["SENTIDX_COL"], CFG["ISREQ_COL"], CFG["SEED_COL"], CFG["PAIR_COL"]]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    LABELS = CFG["LABELS"]; label2id = {l:i for i,l in enumerate(LABELS)}
    df["gold_list"] = df[CFG["LABEL_COL"]].map(lambda x: parse_labels(x, LABELS))
    df["y_vec"]     = df["gold_list"].map(lambda names: to_multi_hot(names, label2id))
    return df.reset_index(drop=True)

def build_req_plus_rep_pairs(df: pd.DataFrame, CFG) -> List[Dict[str, Any]]:
    """
    For each (seed, pair_idx), build one long sequence:
      Request sentences first, then Reply sentences.
    Keep per-sentence email_id and sentence_idx for correct export later.
    """
    exs = []
    sort_cols = [CFG["SEED_COL"], CFG["PAIR_COL"], CFG["ISREQ_COL"], CFG["EMAIL_COL"], CFG["SENTIDX_COL"]]
    for (sd, pr), grp in df.sort_values(sort_cols).groupby([CFG["SEED_COL"], CFG["PAIR_COL"]], sort=False):
        req = grp[grp[CFG["ISREQ_COL"]]==1].sort_values([CFG["EMAIL_COL"], CFG["SENTIDX_COL"]])
        rep = grp[grp[CFG["ISREQ_COL"]]==0].sort_values([CFG["EMAIL_COL"], CFG["SENTIDX_COL"]])

        texts_req  = req[CFG["TEXT_COL"]].astype(str).tolist()
        texts_rep  = rep[CFG["TEXT_COL"]].astype(str).tolist()
        labels_req = [np.array(v, dtype=np.float32) for v in req["y_vec"].tolist()]
        labels_rep = [np.array(v, dtype=np.float32) for v in rep["y_vec"].tolist()]
        sidx_req   = req[CFG["SENTIDX_COL"]].astype(int).tolist()
        sidx_rep   = rep[CFG["SENTIDX_COL"]].astype(int).tolist()
        eid_req    = req[CFG["EMAIL_COL"]].astype(int).tolist()
        eid_rep    = rep[CFG["EMAIL_COL"]].astype(int).tolist()

        texts  = texts_req + texts_rep
        labels = labels_req + labels_rep
        sides  = [0]*len(texts_req) + [1]*len(texts_rep)   # 0=Request, 1=Reply
        sentidxs = sidx_req + sidx_rep
        emailids = eid_req + eid_rep

        exs.append({
            "seed": int(sd), "pair_idx": int(pr),
            "texts": texts, "labels": labels, "sides": sides,
            "sentence_idx_list": sentidxs,
            "email_id_list": emailids,
        })
    return exs

def concat_with_ranges(texts: List[str]):
    parts, ranges, pos = [], [], 0
    for i, s in enumerate(texts):
        if i > 0:
            parts.append(" "); pos += 1
        start = pos
        parts.append(s); pos += len(s)
        ranges.append((start, pos))
    return "".join(parts), ranges

def encode_with_sentence_ids_and_types(concat_text, char_ranges, side_flags, tokenizer, max_length):
    enc = tokenizer(
        concat_text,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        truncation=True, max_length=max_length, add_special_tokens=True, padding=False
    )
    ids, attn = enc["input_ids"], enc["attention_mask"]
    offsets, sp = enc["offset_mapping"], enc["special_tokens_mask"]
    L = len(ids)
    token_type_ids = [0]*L
    S = len(char_ranges)
    sentence_ids = [-1]*L
    sent_token_counts = [0]*S

    for t_idx, ((a,b), is_sp, m) in enumerate(zip(offsets, sp, attn)):
        if m == 0 or is_sp or a == b:  # pad/special/empty
            continue
        for s_idx, (sa,se) in enumerate(char_ranges):
            if not (b <= sa or a >= se):
                sentence_ids[t_idx] = s_idx
                token_type_ids[t_idx] = int(side_flags[s_idx])  # 0=Req,1=Rep
                sent_token_counts[s_idx] += 1
                break

    kept = [i for i,c in enumerate(sent_token_counts) if c>0]
    if not kept:
        return ids, attn, token_type_ids, sentence_ids, kept, []
    remap = {old:i for i,old in enumerate(kept)}
    sentence_ids = [(remap[s] if (s!=-1 and s in remap) else -1) for s in sentence_ids]
    side_kept = [side_flags[i] for i in kept]
    return ids, attn, token_type_ids, sentence_ids, kept, side_kept

class SeqLabelReqPlusRepDataset(Dataset):
    """
    Each item is a concatenated (Request sentences + Reply sentences) sequence
    with per-token sentence_ids for mean pooling into per-sentence embeddings.
    """
    def __init__(self, pairs, tokenizer, cfg):
        self.examples=[]
        self._skipped=0
        for ex in pairs:
            concat, ranges = concat_with_ranges(ex["texts"])
            ids, attn, tt, sids, kept, side_kept = encode_with_sentence_ids_and_types(
                concat, ranges, ex["sides"], tokenizer, cfg["max_length"]
            )
            if len(kept)==0:
                self._skipped+=1; continue
            labels_kept   = [ex["labels"][i] for i in kept]
            sidxs_kept    = [ex["sentence_idx_list"][i] for i in kept]
            emailids_kept = [ex["email_id_list"][i]   for i in kept]
            self.examples.append({
                "input_ids": ids, "attention_mask": attn, "token_type_ids": tt,
                "sentence_ids": sids, "labels": labels_kept, "side_flags": side_kept,
                "seed": ex["seed"], "pair_idx": ex["pair_idx"],
                "sentidx_list": sidxs_kept, "emailid_list": emailids_kept,
                "concat_text": concat,
            })
        if self._skipped:
            print(f"[WARN] Dropped {self._skipped} sequences with zero surviving sentences after truncation.")

    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        e = self.examples[idx]
        labels = torch.tensor(np.stack(e["labels"], axis=0), dtype=torch.float32)
        return {
            "input_ids": torch.tensor(e["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(e["attention_mask"], dtype=torch.long),
            "token_type_ids": torch.tensor(e["token_type_ids"], dtype=torch.long),
            "sentence_ids": torch.tensor(e["sentence_ids"], dtype=torch.long),
            "labels": labels,  # [S, C]
            "side_flags": torch.tensor(e["side_flags"], dtype=torch.long),
        }

class DataCollatorSeqLabelSentenceID:
    def __init__(self, pad_token_id, label_dim): self.pad_token_id=pad_token_id; self.label_dim=label_dim
    def __call__(self, batch):
        max_len = max(x["input_ids"].shape[0] for x in batch)
        input_ids, attention_mask, token_type_ids, sentence_ids = [], [], [], []
        for x in batch:
            pad = max_len - x["input_ids"].shape[0]
            input_ids.append(      F.pad(x["input_ids"],      (0,pad), value=self.pad_token_id))
            attention_mask.append( F.pad(x["attention_mask"], (0,pad), value=0))
            token_type_ids.append( F.pad(x["token_type_ids"], (0,pad), value=0))
            sentence_ids.append(   F.pad(x["sentence_ids"],   (0,pad), value=-1))
        input_ids      = torch.stack(input_ids, dim=0)
        attention_mask = torch.stack(attention_mask, dim=0)
        token_type_ids = torch.stack(token_type_ids, dim=0)
        sentence_ids   = torch.stack(sentence_ids, dim=0)

        max_S = max(x["labels"].shape[0] for x in batch)
        label_grid = []
        for x in batch:
            S = x["labels"].shape[0]
            if S < max_S:
                pad_rows = torch.full((max_S - S, self.label_dim), -1.0, dtype=torch.float32)
                label_grid.append(torch.cat([x["labels"], pad_rows], dim=0))
            else:
                label_grid.append(x["labels"])
        labels = torch.stack(label_grid, dim=0)
        return {"input_ids":input_ids,"attention_mask":attention_mask,"token_type_ids":token_type_ids,
                "sentence_ids":sentence_ids,"labels":labels}

# ============================ FA model (ASL) ============================

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=4.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gp, self.gn, self.clip, self.eps = gamma_pos, gamma_neg, clip, eps
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        # positive term
        log_pos = torch.log(p.clamp(min=self.eps))
        if self.gp > 0:
            log_pos = log_pos * (1.0 - p) ** self.gp
        # negative term with clip on p (not on 1-p)
        pn = (p - self.clip).clamp_min(0.0) if (self.clip and self.clip > 0) else p
        log_neg = torch.log((1.0 - pn).clamp(min=self.eps))
        if self.gn > 0:
            log_neg = log_neg * (pn ** self.gn)
        loss = -(targets * log_pos + (1.0 - targets) * log_neg)
        return loss.mean()

class BertSeqLabelSentenceID(nn.Module):
    def __init__(self, model_name: str, num_labels: int, asl_gp=0.0, asl_gn=4.0, asl_clip=0.05):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.asl = AsymmetricLoss(asl_gp, asl_gn, asl_clip)

    @staticmethod
    def _mean_pool_by_sid(H_b, attn_b, sid_b, S_b, Hdim):
        valid = (attn_b > 0) & (sid_b >= 0)
        if valid.sum() == 0 or S_b == 0:
            return torch.zeros(S_b, Hdim, device=H_b.device)
        idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        sid = sid_b[idx]; feats = H_b[idx, :]
        sums = torch.zeros(S_b, Hdim, device=H_b.device); sums.index_add_(0, sid, feats)
        counts = torch.bincount(sid, minlength=S_b).unsqueeze(1).clamp_min(1)
        return sums / counts

    def forward(self, input_ids, attention_mask, token_type_ids=None, sentence_ids=None, labels=None):
        use_tti = getattr(self.bert.config, "type_vocab_size", 0) > 0
        tti = token_type_ids if use_tti else None

        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=tti)
        H = out.last_hidden_state
        B, L, Hdim = H.shape
        S_max = labels.shape[1] if labels is not None else (sentence_ids.max(dim=1).values.clamp_min(-1).max().item() + 1)
        sent_embs = []
        for b in range(B):
            S_b = int((labels[b, :, 0] != -1).sum().item()) if labels is not None else int(sentence_ids[b].max().item() + 1)
            means_b = self._mean_pool_by_sid(H[b], attention_mask[b], sentence_ids[b], S_b, Hdim)
            if S_b < S_max:
                pad = torch.zeros(S_max - S_b, Hdim, device=H.device); means_b = torch.cat([means_b, pad], dim=0)
            sent_embs.append(means_b)
        sent_embs = torch.stack(sent_embs, dim=0)
        logits = self.classifier(self.dropout(sent_embs))
        loss = None
        if labels is not None:
            mask = (labels[..., 0] != -1)
            if mask.any():
                y = labels[mask, :]; z = logits[mask, :]
                loss = self.asl(z, y)
            else:
                loss = torch.zeros([], device=logits.device)
        return {"loss": loss, "logits": logits}

# ============================ FA metrics helpers (overflow-safe) ============================

def flatten_valid(logits: np.ndarray, labels: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    padded_mask = (labels == -1).all(axis=-1)
    keep_mask = ~padded_mask
    probs2d = logits.reshape(-1, logits.shape[-1])[keep_mask.reshape(-1)]
    gold2d  = labels.reshape(-1, labels.shape[-1])[keep_mask.reshape(-1)]
    gold2d = (gold2d > 0).astype(int)
    return probs2d, gold2d

def metrics_from_bin(gold: np.ndarray, pred: np.ndarray) -> Dict[str,float]:
    hamming = np.not_equal(pred, gold).mean()
    avg_true_k = float(gold.sum(axis=1).mean())
    avg_pred_k = float(pred.sum(axis=1).mean())
    return {
        "micro/f1":        f1_score(gold, pred, average="micro",  zero_division=0),
        "macro/f1":        f1_score(gold, pred, average="macro",  zero_division=0),
        "micro/precision": precision_score(gold, pred, average="micro", zero_division=0),
        "micro/recall":    recall_score(gold, pred, average="micro",  zero_division=0),
        "jaccard/micro":   jaccard_score(gold, pred, average="micro",  zero_division=0),
        "jaccard/macro":   jaccard_score(gold, pred, average="macro",  zero_division=0),
        "jaccard/samples": jaccard_score(gold, pred, average="samples", zero_division=0),
        "hamming_loss":    float(hamming),
        "avg_true_k":      avg_true_k,
        "avg_pred_k":      avg_pred_k,
    }

def training_metrics_fixed_tau(eval_pred):
    logits = getattr(eval_pred, "predictions", None)
    labels = getattr(eval_pred, "label_ids", None)
    if logits is None or labels is None:
        logits, labels = eval_pred
    probs2d, gold2d = flatten_valid(logits, labels)
    preds = (sigmoid_stable_np(probs2d) >= 0.50).astype(int)
    return metrics_from_bin(gold2d, preds)

# ============================ FA training & prediction (Req+Rep) ============================

def train_fa_req_plus_rep(tr_pairs, va_pairs, te_pairs, tok, CFG, LABELS, out_dir, fast=False, num_workers=4):
    label_dim = len(LABELS)
    collator = DataCollatorSeqLabelSentenceID(
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else 0,
        label_dim=label_dim
    )
    ds_tr = SeqLabelReqPlusRepDataset(tr_pairs, tok, CFG)
    ds_va = SeqLabelReqPlusRepDataset(va_pairs, tok, CFG)
    ds_te = SeqLabelReqPlusRepDataset(te_pairs, tok, CFG)

    model = BertSeqLabelSentenceID(
        CFG["fa_model_name"], num_labels=label_dim,
        asl_gp=CFG["asl_gamma_pos"], asl_gn=CFG["asl_gamma_neg"], asl_clip=CFG["asl_clip"]
    )

    has_cuda = torch.cuda.is_available()
    try:
        bf16_ok = has_cuda and torch.cuda.is_bf16_supported()
    except Exception:
        bf16_ok = False
    fp16_ok = has_cuda and not bf16_ok

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=CFG["fa_batch_size"],
        per_device_eval_batch_size=CFG["fa_batch_size"],
        learning_rate=CFG["fa_lr"],
        weight_decay=CFG["fa_weight_decay"],
        num_train_epochs=CFG["fa_epochs"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="micro/f1",
        greater_is_better=True,
        fp16=fp16_ok,
        bf16=bf16_ok,
        seed=CFG["seed"],
        report_to=[],
        logging_steps=100 if not fast else 200,
        remove_unused_columns=False,
        dataloader_num_workers=num_workers if not fast else max(1, num_workers//2),
        dataloader_pin_memory=True,
        save_total_limit=1

    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=ds_tr, eval_dataset=ds_va,
        tokenizer=tok, data_collator=collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG["fa_early_stop"])],
        compute_metrics=training_metrics_fixed_tau
    )
    print("SeqLabel + ASL — Request+Reply (overflow-safe metrics)")
    trainer.train()

    # ---- VAL threshold info (for reporting)
    val_out = trainer.predict(ds_va)
    tau_f1 = float(CFG.get("fa_force_tau_f1", 0.60))
    print(f"[VAL] Reporting τ@F1 with τ={tau_f1:.2f}")
    print(json.dumps(_eval_with_tau(val_out.predictions, val_out.label_ids, tau_f1), indent=2))

    # ---- TEST predictions (returned to caller via ds_te)
    te_out = trainer.predict(ds_te)
    _ = te_out.predictions  # not used directly here

    return trainer, ds_tr, ds_va, ds_te, te_out.predictions, tau_f1

def _eval_with_tau(logits, labels, tau):
    probs2d, gold2d = flatten_valid(logits, labels)
    pred = (sigmoid_stable_np(probs2d) >= float(tau)).astype(int)
    return metrics_from_bin(gold2d, pred)

def predict_sentence_probs_req_plus_rep(ds, trainer):
    out = trainer.predict(ds)
    logits = out.predictions
    probs = sigmoid_stable_np(logits)
    return probs, logits

def predict_fa_to_rows_req_plus_rep(ds, probs, LABELS, DOC_ID_COL, SENTIDX_COL):
    """
    Export per sentence: email_id, sentence_idx, prob_* — EVEN THOUGH training used paired sequences.
    """
    rows = []
    for ex_i, ex in enumerate(ds.examples):
        S = len(ex["labels"])
        P = probs[ex_i][:S]
        for j in range(S):
            row = {DOC_ID_COL: str(ex["emailid_list"][j]), SENTIDX_COL: int(ex["sentidx_list"][j])}
            for k, lab in enumerate(LABELS):
                row[f"prob_{lab}"] = float(P[j, k])
            rows.append(row)
    return rows

# ============================ PredFA aggregation (HSPT-13) ============================

def aggregate_predfa_hspt13(dfp: pd.DataFrame, DOC_ID_COL: str, LABELS: List[str], SENTIDX_COL: str):
    """
    Build document-level PredFA features: HSPT-13
      1-9:  sumprob_<label> (sum over sentences of probabilities for each FA label; includes Neutral)
     10-11: sum_H, sum_S            (sum of all H* vs all S* probs; Neutral excluded)
     12-13: sum_praise, sum_threat  (sum of all '+' vs all '−' probs; Neutral excluded)
    """
    req = [DOC_ID_COL, SENTIDX_COL] + [f"prob_{l}" for l in LABELS]
    for c in req: assert c in dfp.columns, f"Missing '{c}' in PredFA DF"

    dfp = dfp.copy().sort_values([DOC_ID_COL, SENTIDX_COL])

    # define groups (Neutral excluded where noted)
    H_labels = ["HNeg+","HNeg-","HPos+","HPos-"]
    S_labels = ["SNeg+","SNeg-","SPos+","SPos-"]
    plus_labels  = ["HPos+","HNeg+","SPos+","SNeg+"]
    minus_labels = ["HPos-","HNeg-","SPos-","SNeg-"]

    lab_cols = [f"prob_{l}" for l in LABELS]
    rows = []
    for doc_id, g in dfp.groupby(DOC_ID_COL, sort=False):
        probs = _scrub(g[lab_cols].to_numpy(np.float32))  # [n_sent, 9]
        sums = probs.sum(axis=0)  # [9]
        row = {DOC_ID_COL: str(doc_id)}
        # 1-9 per-label sums
        for i, lab in enumerate(LABELS):
            row[f"sumprob_{lab}"] = float(sums[i])
        # H/S (exclude Neutral)
        sum_H = float(g[[f"prob_{l}" for l in H_labels]].sum().sum())
        sum_S = float(g[[f"prob_{l}" for l in S_labels]].sum().sum())
        # praising (+) vs threatening (−) (exclude Neutral)
        sum_praise = float(g[[f"prob_{l}" for l in plus_labels]].sum().sum())
        sum_threat = float(g[[f"prob_{l}" for l in minus_labels]].sum().sum())
        row.update({"sum_H":sum_H, "sum_S":sum_S, "sum_praise":sum_praise, "sum_threat":sum_threat})
        rows.append(row)

    # canonical order
    hspt_cols = [f"sumprob_{lab}" for lab in LABELS] + ["sum_H","sum_S","sum_praise","sum_threat"]
    out = pd.DataFrame(rows)
    return out[[DOC_ID_COL] + hspt_cols]

def standardize_predfa_features(F_tr, F_va, F_te, doc_id_col="email_id"):
    """
    Z-score all PredFA features (sumprob_* + sum_H + sum_S + sum_praise + sum_threat) with TRAIN statistics.
    """
    def _slim(F):
        keep = [c for c in F.columns if (c == doc_id_col
                                         or c.startswith("sumprob_")
                                         or c in ("sum_H","sum_S","sum_praise","sum_threat"))]
        F = F[keep].copy()
        for c in F.columns:
            if c != doc_id_col:
                F[c] = pd.to_numeric(F[c], errors="coerce").fillna(0.0)
        return F

    F_tr = _slim(F_tr); F_va = _slim(F_va); F_te = _slim(F_te)

    feat_cols = [c for c in F_tr.columns if c != doc_id_col]
    mu = F_tr[feat_cols].mean(0)
    sd = F_tr[feat_cols].std(0).replace(0, 1.0)

    for F in (F_tr, F_va, F_te):
        F.loc[:, feat_cols] = (F[feat_cols] - mu) / sd

    return F_tr, F_va, F_te

# ============================ Doc-level models ============================

class BertDocRegressor(nn.Module):
    """
    Concatenation-only late fusion:
      z = [h_doc ; f_predfa]
      h = Linear(in_dim -> 128) + ReLU
      y_t = Linear(128 -> 1)  for each target t
    """
    def __init__(self, encoder_name="bert-base-uncased", n_targets=3, fusion_dim=0, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hid = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.fusion_dim = fusion_dim

        in_dim = hid + (fusion_dim if fusion_dim > 0 else 0)

        # EXACT requirement:  ... → 128 with ReLU
        self.hidden = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        # EXACT requirement:  128 → 1 (per target)
        self.heads = nn.ModuleList([nn.Linear(128, 1) for _ in range(n_targets)])

    def forward(self, input_ids, attention_mask, n_chunks, predfa=None):
        B, K, L = input_ids.shape
        ids = input_ids.view(B*K, L); att = attention_mask.view(B*K, L)
        out = self.encoder(input_ids=ids, attention_mask=att, return_dict=True)
        pooled = out.pooler_output if (hasattr(out, "pooler_output") and out.pooler_output is not None) else out.last_hidden_state[:,0,:]
        H = pooled.size(-1); pooled = pooled.view(B, K, H)

        mask = (torch.arange(K, device=n_chunks.device).unsqueeze(0) < n_chunks.unsqueeze(1)).float().unsqueeze(-1)
        doc_vec = (pooled * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

        # Concatenation-only fusion (NO LayerNorm, NO gate)
        if (predfa is not None) and (self.fusion_dim > 0):
            doc_vec = torch.cat([doc_vec, predfa], dim=-1)

        doc_vec = self.dropout(doc_vec)
        h = self.hidden(doc_vec)  # [B,128]

        outs = [head(h) for head in self.heads]   # list of [B,1]
        return torch.cat(outs, dim=-1)            # [B, n_targets]

# -------- Tokenization Cache for DocDataset --------
_DOC_TOKEN_CACHE: Dict[Tuple[str,str,int,int], Tuple[List[List[int]], List[List[int]]]] = {}

def _hash_text(text: str) -> str:
    return hashlib.md5(text.encode('utf-8')).hexdigest()

class DocDataset(Dataset):
    def __init__(self, df, text_col, targets, tokenizer, mode="simple",
                 max_len=512, chunk_len=400, chunk_stride=350, predfa_df=None, id_col="email_id"):
        self.df = df.reset_index(drop=True); self.text_col=text_col; self.targets=targets; self.tk=tokenizer
        self.mode=mode; self.max_len=max_len; self.chunk_len=chunk_len; self.chunk_stride=chunk_stride
        self.id_col = id_col; self.predfa=None; self.predfa_cols=[]
        self.encoder_id = getattr(self.tk, "name_or_path", "tokenizer")
        if predfa_df is not None and id_col in predfa_df.columns:
            predfa_df = predfa_df.set_index(id_col)
            self.predfa = predfa_df.reindex(self.df[id_col].astype(str).values).reset_index(drop=True).fillna(0.0)
            self.predfa_cols = [c for c in self.predfa.columns if c != id_col]
    def __len__(self): return len(self.df)
    def _encode_chunks(self, text):
        key = (self.encoder_id, _hash_text(text), self.chunk_len, self.chunk_stride)
        if key in _DOC_TOKEN_CACHE:
            toks_list, att_list = _DOC_TOKEN_CACHE[key]
            return toks_list, att_list
        toks = self.tk.encode(text, add_special_tokens=False, truncation=False)
        if len(toks)==0: toks=[self.tk.unk_token_id]
        toks_list=[]; att_list=[]; i=0
        while i < len(toks):
            win = toks[i:i+self.chunk_len]
            win = [self.tk.cls_token_id] + win + [self.tk.sep_token_id]
            if len(win) > self.max_len:
                win = win[:self.max_len]
                if win[-1] != self.tk.sep_token_id: win[-1]=self.tk.sep_token_id
            attn = [1]*len(win)
            toks_list.append(win); att_list.append(attn)
            if i + self.chunk_len >= len(toks): break
            i += self.chunk_stride
        _DOC_TOKEN_CACHE[key] = (toks_list, att_list)
        return toks_list, att_list
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; text = str(row[self.text_col]) if pd.notna(row[self.text_col]) else ""
        y = row[self.targets].values.astype(np.float32)
        if self.mode=="simple":
            enc = self.tk(text, max_length=self.max_len, truncation=True, padding='max_length', return_tensors='pt')
            ids = enc['input_ids'].squeeze(0).unsqueeze(0); att = enc['attention_mask'].squeeze(0).unsqueeze(0); K=1
        else:
            chunks, atts = self._encode_chunks(text)
            L=max(len(c) for c in chunks)
            ids=[]; att=[]
            for w,a in zip(chunks, atts):
                pad=L-len(w); ids.append(w+[self.tk.pad_token_id]*pad); att.append(a+[0]*pad)
            ids=torch.tensor(ids, dtype=torch.long); att=torch.tensor(att, dtype=torch.long); K=ids.size(0)
        item={"input_ids":ids, "attention_mask":att, "targets":torch.tensor(y, dtype=torch.float32), "n_chunks":K}
        if self.predfa is not None:
            feats = self.predfa.iloc[idx][self.predfa_cols].values.astype(np.float32)
            feats = _scrub(feats)
            item["predfa"]=torch.tensor(feats, dtype=torch.float32)
        else:
            item["predfa"]=None
        return item

def doc_collate(batch):
    max_K = max(b["input_ids"].size(0) for b in batch)
    L = max(b["input_ids"].size(1) for b in batch)
    def pad2(t, fill=0):
        if t.size(1) < L:
            pad_len = L - t.size(1)
            t = torch.cat([t, torch.full((t.size(0), pad_len), fill, dtype=t.dtype)], dim=1)
        if t.size(0) < max_K:
            pad_k = torch.full((max_K - t.size(0), L), fill, dtype=t.dtype)
            t = torch.cat([t, pad_k], dim=0)
        return t
    ids   = torch.stack([pad2(b["input_ids"], fill=0) for b in batch], dim=0)
    att   = torch.stack([pad2(b["attention_mask"], fill=0) for b in batch], dim=0)
    y     = torch.stack([b["targets"] for b in batch], dim=0)
    nk    = torch.tensor([b["n_chunks"] for b in batch], dtype=torch.long)

    predf_list = [b["predfa"] for b in batch]
    if all(p is None for p in predf_list):
        P = None
    else:
        maxD = max((p.numel() if p is not None else 0) for p in predf_list)
        rows=[]
        for p in predf_list:
            if p is None:
                rows.append(torch.zeros(maxD, dtype=torch.float32))
            elif p.numel() < maxD:
                rows.append(torch.cat([p, torch.zeros(maxD - p.numel())], dim=0))
            else:
                rows.append(p)
        P = torch.stack(rows, dim=0)
    return {"input_ids": ids, "attention_mask": att, "targets": y, "n_chunks": nk, "predfa": P}

# ============================ Trainers (BERT / PredFA MLP) ============================

def train_eval_doc_bert(df_tr, df_va, df_te, CFG, text_col, targets, predfa_agg=None,
                        mode="simple", outdir="./doc_bert", use_predfa=False,
                        multi_task=True, num_workers=4):
    ensure_dir(outdir)
    tk = AutoTokenizer.from_pretrained(CFG["bert_encoder"], use_fast=True)
    device=_safe_device()
    lam = CFG.get("doc_early_stop_lambda", 1e-3)
    ID_COL = CFG["DOC_ID_COL"]

    def run_once(trg_list, outdir_run):
        ensure_dir(outdir_run)
        global_best_score = -1e9
        global_best_state = None
        global_best_cfg   = None

        for lr in CFG["bert_lr_grid"]:
            for ep in CFG["bert_epochs_grid"]:
                for dr in CFG["bert_dropout_grid"]:
                    for bs in CFG["bert_batch_grid"]:
                        for chL in ([None] if mode=="simple" else CFG["chunk_len_grid"]):
                            for chS in ([None] if mode=="simple" else CFG["chunk_stride_grid"]):
                                print(f"[BERT] try lr={lr} ep={ep} drop={dr} bs={bs} mode={mode} chunk=({chL},{chS}) predfa={use_predfa}")
                                ds_tr = DocDataset(df_tr, text_col, trg_list, tk, mode=mode,
                                                   max_len=512, chunk_len=chL or 400, chunk_stride=chS or 350,
                                                   predfa_df=predfa_agg if use_predfa else None,
                                                   id_col=ID_COL)
                                ds_va = DocDataset(df_va, text_col, trg_list, tk, mode=mode,
                                                   max_len=512, chunk_len=chL or 400, chunk_stride=chS or 350,
                                                   predfa_df=predfa_agg if use_predfa else None,
                                                   id_col=ID_COL)
                                dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True,  collate_fn=doc_collate,
                                                   pin_memory=True, num_workers=num_workers)
                                dl_va = DataLoader(ds_va, batch_size=bs, shuffle=False, collate_fn=doc_collate,
                                                   pin_memory=True, num_workers=num_workers)

                                fusion_dim = 0
                                if use_predfa and predfa_agg is not None:
                                    sample = ds_tr[0]
                                    fusion_dim = sample["predfa"].numel()

                                model = BertDocRegressor(encoder_name=CFG["bert_encoder"], n_targets=len(trg_list),
                                                         fusion_dim=fusion_dim, dropout=dr).to(device)
                                opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
                                total_steps = max(1, len(dl_tr)*ep)
                                warmup_steps = int(0.06*total_steps)
                                sch = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)
                                loss_fn = nn.SmoothL1Loss()

                                patience = CFG.get("doc_early_stop_patience", 2)
                                min_delta = CFG.get("doc_early_stop_min_delta", 1e-4)
                                best_val = -1e9
                                best_state = None
                                bad_epochs = 0

                                for epoch in range(1, ep+1):
                                    model.train()
                                    for batch in dl_tr:
                                        opt.zero_grad()
                                        ids=batch["input_ids"].to(device); att=batch["attention_mask"].to(device)
                                        tg=batch["targets"].to(device); nk=batch["n_chunks"].to(device)
                                        pf=batch["predfa"].to(device) if batch["predfa"] is not None else None
                                        preds = model(ids, att, nk, pf); loss=loss_fn(preds, tg)
                                        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                                        opt.step(); sch.step()

                                    model.eval(); Ys=[]; Ps=[]
                                    with torch.no_grad():
                                        for batch in dl_va:
                                            ids=batch["input_ids"].to(device); att=batch["attention_mask"].to(device)
                                            tg=batch["targets"].to(device); nk=batch["n_chunks"].to(device)
                                            pf=batch["predfa"].to(device) if batch["predfa"] is not None else None
                                            preds = model(ids, att, nk, pf)
                                            Ys.append(tg.cpu().numpy()); Ps.append(preds.cpu().numpy())
                                    Yv=np.concatenate(Ys,0); Pv=np.concatenate(Ps,0)
                                    rho_macro, rhos = spearman_macro(Yv, Pv)
                                    mae_val, _ = mae_rmse(Yv, Pv)
                                    score = rho_macro - lam * float(np.mean(mae_val))

                                    if epoch == 1 and best_val == -1e9 and score < -0.5:
                                        print("[BERT] Prune candidate (weak after epoch 1).")
                                        break

                                    if score > best_val + min_delta:
                                        best_val = score
                                        bad_epochs = 0
                                        best_state = {
                                            "model": model.state_dict(),
                                            "cfg": {"lr":lr,"epochs":epoch,"dropout":dr,"batch":bs,"mode":mode,
                                                    "chunk_len":chL,"chunk_stride":chS,"use_predfa":use_predfa,
                                                    "targets":trg_list}
                                        }
                                    else:
                                        bad_epochs += 1
                                        if bad_epochs >= patience:
                                            print(f"[BERT] Early stop at epoch {epoch} (best score={best_val:.4f}).")
                                            break

                                if best_state is not None and best_val > global_best_score:
                                    global_best_score = best_val
                                    global_best_state = best_state
                                    global_best_cfg   = best_state["cfg"]

        if global_best_state is None:
            raise RuntimeError("BERT doc tuning produced no runs (check dataset sizes).")

        ds_te = DocDataset(
            df_te, text_col, trg_list, tk, mode=global_best_cfg["mode"],
            max_len=512, chunk_len=global_best_cfg["chunk_len"] or 400,
            chunk_stride=global_best_cfg["chunk_stride"] or 350,
            predfa_df=predfa_agg if global_best_cfg["use_predfa"] else None, id_col=ID_COL
        )
        fusion_dim = 0
        if global_best_cfg["use_predfa"] and predfa_agg is not None:
            sample = ds_te[0]; fusion_dim = sample["predfa"].numel()

        model = BertDocRegressor(encoder_name=CFG["bert_encoder"], n_targets=len(trg_list),
                                 fusion_dim=fusion_dim, dropout=global_best_cfg["dropout"]).to(device)
        model.load_state_dict(global_best_state["model"])
        model.eval()

        dl_te = DataLoader(ds_te, batch_size=global_best_cfg["batch"], shuffle=False,
                           collate_fn=doc_collate, pin_memory=True, num_workers=num_workers)
        Ys=[]; Ps=[]
        with torch.no_grad():
            for batch in dl_te:
                ids=batch["input_ids"].to(device); att=batch["attention_mask"].to(device)
                tg=batch["targets"].to(device); nk=batch["n_chunks"].to(device)
                pf=batch["predfa"].to(device) if batch["predfa"] is not None else None
                preds = model(ids, att, nk, pf); Ys.append(tg.cpu().numpy()); Ps.append(preds.cpu().numpy())
        Yt=np.concatenate(Ys,0); Pt=np.concatenate(Ps,0)
        mae_te, rmse_te = mae_rmse(Yt, Pt)
        rhos_te = spearman_each(Yt, Pt)
        rho_macro = float(np.mean(rhos_te))

        ensure_dir(outdir_run)
        with open(os.path.join(outdir_run, "metrics.json"), "w") as f:
            json.dump({
                "best_cfg": global_best_cfg,
                "dev_best_score": global_best_score,
                "test": {"MAE": mae_te.tolist(), "RMSE": rmse_te.tolist(),
                         "rho_macro": rho_macro, "rhos": rhos_te},
                "targets": trg_list
            }, f, indent=2)
        np.save(os.path.join(outdir_run,"test_y.npy"), Yt)
        np.save(os.path.join(outdir_run,"test_pred.npy"), Pt)
        return Yt, Pt, mae_te.tolist(), rho_macro, rhos_te

    if multi_task:
        Y, P, MAE, rho_macro, rhos = run_once(targets, outdir+"_mt")
        return {"Y":Y, "P":P, "MAE":MAE, "rho_macro":rho_macro, "rhos":rhos, "variant":"mt"}

    all_P=[]; all_Y=None; maes=[]; rhos=[]
    for ti, tgt in enumerate(targets):
        Yt, Pt, MAE, rho, rhos_single = run_once([tgt], outdir+"_st_"+str(ti))
        if all_Y is None: all_Y = Yt
        all_P.append(Pt)
        maes.append(MAE[0]); rhos.append(rhos_single[0])
    P = np.concatenate(all_P, axis=1)
    rho_macro = float(np.mean(rhos))
    ensure_dir(outdir+"_st")
    with open(os.path.join(outdir+"_st", "metrics.json"), "w") as f:
        json.dump({
            "best_cfg": "per-target single-task (see *_st_0,1,2)",
            "test": {"MAE": maes, "rho_macro": rho_macro, "rhos": rhos},
            "targets": targets
        }, f, indent=2)
    np.save(os.path.join(outdir+"_st","test_y.npy"), all_Y)
    np.save(os.path.join(outdir+"_st","test_pred.npy"), P)
    return {"Y":all_Y, "P":P, "MAE":maes, "rho_macro":rho_macro, "rhos":rhos, "variant":"st"}

# -------------------- PredFA-only MLP --------------------

class PredFAMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x): return self.net(x)

def run_predfa_mlp(df_tr_doc, df_va_doc, df_te_doc, F_tr, F_va, F_te, targets, CFG, outdir, multi_task=True):
    def align(df_doc, F):
        F = F.copy()
        assert CFG["DOC_ID_COL"] in F.columns
        return F.set_index(CFG["DOC_ID_COL"]).reindex(df_doc[CFG["DOC_ID_COL"]].astype(str).values).fillna(0.0).reset_index(drop=True)

    ensure_dir(outdir)
    feat_cols = [c for c in F_tr.columns if c != CFG["DOC_ID_COL"]]

    def _train_epoch_loop(model, Xtr, Ytr, Xva, Yva, device, lr, epochs, patience, min_delta):
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        loss_fn = nn.SmoothL1Loss()
        idx = torch.arange(Xtr.size(0))

        best_score = -1e9
        best_state = None
        bad_epochs = 0
        lam = CFG.get("doc_early_stop_lambda", 1e-3)

        for ep in range(1, epochs+1):
            model.train()
            perm = idx[torch.randperm(idx.numel())]
            for i in range(0, perm.numel(), CFG["mlp_batch"]):
                b = perm[i:i+CFG["mlp_batch"]]
                xb = Xtr[b].to(device); yb=Ytr[b].to(device)
                opt.zero_grad(); pred = model(xb); loss = loss_fn(pred, yb)
                loss.backward(); opt.step()

            model.eval()
            with torch.no_grad():
                Pv = model(Xva.to(device)).cpu().numpy()
            rho_macro, _ = spearman_macro(Yva.numpy(), Pv)
            mae_val, _ = mae_rmse(Yva.numpy(), Pv)
            score = rho_macro - lam * float(np.mean(mae_val))

            if score > best_score + min_delta:
                best_score = score
                best_state = model.state_dict()
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"[MLP] Early stop at epoch {ep} (best score={best_score:.4f}).")
                    break

        return best_state if best_state is not None else model.state_dict(), best_score

    def _train_or_retry_then_cpu(model, Xtr, Ytr, Xva, Yva, device, lr, epochs, patience, min_delta):
        tried_cuda_retry = False
        try:
            return _train_epoch_loop(model.to(device), Xtr, Ytr, Xva, Yva, device, lr, epochs, patience, min_delta)
        except RuntimeError as e:
            if ("CUDA" in str(e)) and (device=="cuda") and (not tried_cuda_retry):
                print("[WARN] CUDA hiccup in PredFA MLP — retrying once on CUDA.")
                tried_cuda_retry = True
                try: torch.cuda.empty_cache()
                except Exception: pass
                return _train_epoch_loop(model.to("cuda"), Xtr, Ytr, Xva, Yva, "cuda", lr, epochs, patience, min_delta)
            print("[WARN] Falling back to CPU for PredFA MLP.")
            return _train_epoch_loop(model.to("cpu"), Xtr, Ytr, Xva, Yva, "cpu", lr, epochs, patience, min_delta)

    def run_once(trg_list, outdir_run):
        ensure_dir(outdir_run)
        Ft = _scrub(align(df_tr_doc, F_tr)[feat_cols].to_numpy(np.float32))
        Fv = _scrub(align(df_va_doc, F_va)[feat_cols].to_numpy(np.float32))
        Fe = _scrub(align(df_te_doc, F_te)[feat_cols].to_numpy(np.float32))
        Yt = _scrub(df_tr_doc[trg_list].to_numpy(np.float32))
        Yv = _scrub(df_va_doc[trg_list].to_numpy(np.float32))
        Ye = _scrub(df_te_doc[trg_list].to_numpy(np.float32))

        Xtr = torch.tensor(Ft); Xva = torch.tensor(Fv); Xte = torch.tensor(Fe)
        Ytr = torch.tensor(Yt); Yva = torch.tensor(Yv); Yte = torch.tensor(Ye)
        best=None; device=_safe_device()

        patience = CFG.get("doc_early_stop_patience", 2)
        min_delta = CFG.get("doc_early_stop_min_delta", 1e-4)

        for hidden in CFG["mlp_hidden_grid"]:
            for dr in CFG["mlp_dropout_grid"]:
                for lr in CFG["mlp_lr_grid"]:
                    model = PredFAMLP(Xtr.size(1), len(trg_list), hidden=hidden, dropout=dr)
                    state, val_score = _train_or_retry_then_cpu(
                        model, Xtr, Ytr, Xva, Yva, device, lr,
                        epochs=CFG["mlp_epochs"], patience=patience, min_delta=min_delta
                    )
                    cand=(val_score, hidden, dr, lr, state)
                    if (best is None) or (val_score>best[0]): best=cand

        _, hidden, dr, lr, state = best
        final_device=_safe_device()
        model = PredFAMLP(Xtr.size(1), len(trg_list), hidden=hidden, dropout=dr).to(final_device)
        model.load_state_dict(state); model.eval()
        with torch.no_grad():
            Pt = model(Xte.to(final_device)).cpu().numpy()
        mae_te, rmse_te = mae_rmse(Ye, Pt)
        rhos_te = spearman_each(Ye, Pt)
        rho_macro = float(np.mean(rhos_te))

        with open(os.path.join(outdir_run, "metrics.json"), "w") as f:
            json.dump({
                "best_cfg": {"hidden": hidden, "dropout": dr, "lr": lr},
                "test": {"MAE": mae_te.tolist(), "RMSE": rmse_te.tolist(),
                         "rho_macro": rho_macro, "rhos": rhos_te},
                "targets": trg_list
            }, f, indent=2)
        np.save(os.path.join(outdir_run,"test_y.npy"), Ye)
        np.save(os.path.join(outdir_run,"test_pred.npy"), Pt)
        return Ye, Pt, mae_te.tolist(), rho_macro, rhos_te

    if multi_task:
        Y, P, MAE, rho_macro, rhos = run_once(targets, outdir+"_mt")
        return {"Y":Y, "P":P, "MAE":MAE, "rho_macro":rho_macro, "rhos":rhos, "variant":"mt"}

    all_P=[]; all_Y=None; maes=[]; rhos=[]
    for ti, tgt in enumerate(targets):
        Yt, Pt, MAE, rho, rhos_single = run_once([tgt], outdir+"_st_"+str(ti))
        if all_Y is None: all_Y = Yt
        all_P.append(Pt); maes.append(MAE[0]); rhos.append(rhos_single[0])
    P = np.concatenate(all_P, axis=1); rho_macro = float(np.mean(rhos))
    ensure_dir(outdir+"_st")
    with open(os.path.join(outdir+"_st", "metrics.json"), "w") as f:
        json.dump({
            "best_cfg": "per-target single-task (see *_st_0,1,2)",
            "test": {"MAE": maes, "rho_macro": rho_macro, "rhos": rhos},
            "targets": targets
        }, f, indent=2)
    np.save(os.path.join(outdir+"_st","test_y.npy"), all_Y)
    np.save(os.path.join(outdir+"_st","test_pred.npy"), P)
    return {"Y":all_Y, "P":P, "MAE":maes, "rho_macro":rho_macro, "rhos":rhos, "variant":"st"}

# ============================ Main ============================

def main():
    ap = argparse.ArgumentParser()
    # paths
    ap.add_argument("--train_csv", type=str, required=True)
    ap.add_argument("--val_csv",   type=str, required=True)
    ap.add_argument("--test_csv",  type=str, required=True)
    ap.add_argument("--doc_csv",   type=str, required=True)
    ap.add_argument("--sent_csv",  type=str, default="")  # unused here; kept for CLI compatibility
    # toggles for which FAMILY to run (both MT & ST will be produced automatically)
    ap.add_argument("--skip_fa", action="store_true")
    ap.add_argument("--do_doc_bert", action="store_true")
    ap.add_argument("--do_predfa_mlp", action="store_true")
    ap.add_argument("--use_predfa_fusion", action="store_true")
    ap.add_argument("--bert_mode", choices=["simple","hier"], default="simple")
    ap.add_argument("--fast", action="store_true", help="Faster run settings (still BERT; smaller grids/lengths)")
    # general
    ap.add_argument("--outdir", type=str, default=DEF_CFG["output_dir"])
    args, _unknown = ap.parse_known_args()

    # ---------- PATH VALIDATION & WRITABLE FALLBACKS ----------
    args.train_csv = _expand(args.train_csv)
    args.val_csv   = _expand(args.val_csv)
    args.test_csv  = _expand(args.test_csv)
    args.doc_csv   = _expand(args.doc_csv)
    for p in [args.train_csv, args.val_csv, args.test_csv, args.doc_csv]:
        _assert_file(p)

    args.outdir = _ensure_writable_dir(args.outdir)
    DEF_CFG["fa_tmpdir"] = _ensure_writable_dir(os.path.join(args.outdir, "fa_models_req_plus_rep"))

    set_seed(DEF_CFG["seed"])
    CFG = DEF_CFG.copy()
    CFG["train_csv"]=args.train_csv; CFG["val_csv"]=args.val_csv; CFG["test_csv"]=args.test_csv
    CFG["DOC_CSV"]=args.doc_csv;    CFG["SENT_CSV"]=args.sent_csv
    CFG["output_dir"]=args.outdir

    # FAST mode tweaks — still BERT, just shorter seqs/smaller grids
    if args.fast:
        print("[FAST MODE] Enabled — BERT kept; shorter sequences, condensed grids.")
        CFG["fa_model_name"] = "bert-base-uncased"
        CFG["bert_encoder"] = "bert-base-uncased"
        CFG["max_length"] = 256
        CFG["fa_epochs"] = 3
        CFG["fa_batch_size"] = 8
        CFG["bert_lr_grid"] = [2e-5]
        CFG["bert_epochs_grid"] = [3]
        CFG["bert_dropout_grid"] = [0.1]
        CFG["bert_batch_grid"] = [8]
        CFG["chunk_len_grid"] = [320]
        CFG["chunk_stride_grid"] = [256]
        CFG["mlp_hidden_grid"] = [128]
        CFG["mlp_dropout_grid"] = [0.1]
        CFG["mlp_lr_grid"] = [1e-3]
        CFG["mlp_epochs"] = 20
        CFG["mlp_batch"] = 512

    # ===== Sentence splits
    df_train = load_sent_split(CFG["train_csv"], CFG)
    df_val   = load_sent_split(CFG["val_csv"],   CFG)
    df_test  = load_sent_split(CFG["test_csv"],  CFG)

    # Guard: no seed overlap
    s_tr, s_va, s_te = set(df_train[CFG["SEED_COL"]]), set(df_val[CFG["SEED_COL"]]), set(df_test[CFG["SEED_COL"]])
    assert not (s_tr & s_va) and not (s_tr & s_te) and not (s_va & s_te), "Leakage: seed overlap across splits"

    LABELS = CFG["LABELS"]; tok = AutoTokenizer.from_pretrained(CFG["fa_model_name"], use_fast=True)
    assert tok.is_fast, "Use a fast tokenizer to get offset_mapping."

    # ===== Phase A: FA → PredFA CSVs (Request+Reply pairing)
    train_pred_csv = os.path.join(CFG["output_dir"], "predfa_train.csv")
    val_pred_csv   = os.path.join(CFG["output_dir"], "predfa_val.csv")
    test_pred_csv  = os.path.join(CFG["output_dir"], "predfa_test.csv")

    if not args.skip_fa:
        print("[FA][Req+Rep] Building paired datasets (TRAIN/VAL/TEST)…")
        tr_pairs = build_req_plus_rep_pairs(df_train, CFG)
        va_pairs = build_req_plus_rep_pairs(df_val,   CFG)
        te_pairs = build_req_plus_rep_pairs(df_test,  CFG)

        print("[FA][Req+Rep] Training on TRAIN, early-stopping on VAL…")
        trainer, ds_tr, ds_va, ds_te, _test_logits_unused, tau_f1 = train_fa_req_plus_rep(
            tr_pairs, va_pairs, te_pairs, tok, CFG, LABELS,
            out_dir=os.path.join(CFG["fa_tmpdir"], "trainval"),
            fast=args.fast, num_workers=CFG["num_workers"]
        )
        print(f"[FA] Reporting with forced Global τ@F1: τ={tau_f1:.2f}")

        # TRAIN preds
        print("[FA] Predicting TRAIN…")
        tr_probs, _ = predict_sentence_probs_req_plus_rep(ds_tr, trainer)
        tr_rows = predict_fa_to_rows_req_plus_rep(ds_tr, tr_probs, LABELS, CFG["DOC_ID_COL"], CFG["SENTIDX_COL"])
        pd.DataFrame(tr_rows).to_csv(train_pred_csv, index=False); print("[FA] Saved:", train_pred_csv)

        # VAL preds
        print("[FA] Predicting VAL…")
        va_probs, _ = predict_sentence_probs_req_plus_rep(ds_va, trainer)
        va_rows = predict_fa_to_rows_req_plus_rep(ds_va, va_probs, LABELS, CFG["DOC_ID_COL"], CFG["SENTIDX_COL"])
        pd.DataFrame(va_rows).to_csv(val_pred_csv, index=False); print("[FA] Saved:", val_pred_csv)

        # TEST preds
        print("[FA] Predicting TEST…")
        te_probs, _ = predict_sentence_probs_req_plus_rep(ds_te, trainer)
        te_rows = predict_fa_to_rows_req_plus_rep(ds_te, te_probs, LABELS, CFG["DOC_ID_COL"], CFG["SENTIDX_COL"])
        pd.DataFrame(te_rows).to_csv(test_pred_csv, index=False); print("[FA] Saved:", test_pred_csv)
    else:
        assert all(os.path.isfile(p) for p in [train_pred_csv, val_pred_csv, test_pred_csv]), \
            "skip_fa set but PredFA CSVs not found."

    # ===== Phase B: Prepare DOC CSV and PredFA features (HSPT-13)
    df_doc_all = pd.read_csv(CFG["DOC_CSV"]).copy()
    for c in [CFG["DOC_ID_COL"], CFG["DOC_TEXT_COL"]] + CFG["TARGETS"]:
        assert c in df_doc_all.columns, f"Missing '{c}' in DOC_CSV"

    # Aggregate PredFA (doc-level) → HSPT-13 + standardize
    F_tr_raw = aggregate_predfa_hspt13(pd.read_csv(train_pred_csv), CFG["DOC_ID_COL"], LABELS, CFG["SENTIDX_COL"])
    F_va_raw = aggregate_predfa_hspt13(pd.read_csv(val_pred_csv),   CFG["DOC_ID_COL"], LABELS, CFG["SENTIDX_COL"])
    F_te_raw = aggregate_predfa_hspt13(pd.read_csv(test_pred_csv),  CFG["DOC_ID_COL"], LABELS, CFG["SENTIDX_COL"])

    # numeric cleaning + z-score
    for F in (F_tr_raw, F_va_raw, F_te_raw):
        for c in F.columns:
            if c != CFG["DOC_ID_COL"]:
                F[c] = pd.to_numeric(F[c], errors="coerce").fillna(0.0)

    F_tr, F_va, F_te = standardize_predfa_features(F_tr_raw, F_va_raw, F_te_raw, doc_id_col=CFG["DOC_ID_COL"])
    for F in (F_tr, F_va, F_te):
        F[CFG["DOC_ID_COL"]] = F[CFG["DOC_ID_COL"]].astype(str)
    df_doc_all[CFG["DOC_ID_COL"]] = df_doc_all[CFG["DOC_ID_COL"]].astype(str)

    # Split doc frames (using seed-disjoint ids derived from sentence splits)
    ids_tr = set(df_train[CFG["EMAIL_COL"]].astype(int).astype(str).unique())
    ids_va = set(df_val  [CFG["EMAIL_COL"]].astype(int).astype(str).unique())
    ids_te = set(df_test [CFG["EMAIL_COL"]].astype(int).astype(str).unique())

    df_tr_doc = df_doc_all[df_doc_all[CFG["DOC_ID_COL"]].isin(ids_tr)].reset_index(drop=True)
    df_va_doc = df_doc_all[df_doc_all[CFG["DOC_ID_COL"]].isin(ids_va)].reset_index(drop=True)
    df_te_doc = df_doc_all[df_doc_all[CFG["DOC_ID_COL"]].isin(ids_te)].reset_index(drop=True)

    # Fusion dataframe (standardized HSPT-13 features; contains email_id)
    predfa_agg_all = pd.concat([F_tr.copy(), F_va.copy(), F_te.copy()], axis=0, ignore_index=True)

    # Sanity checks
    def _check_align(df_doc, F_pred, idcol):
        a = set(df_doc[idcol]); b = set(F_pred[idcol])
        missing = a - b
        assert len(missing) == 0, f"PredFA (HSPT-13) missing {len(missing)} ids; e.g., {list(missing)[:5]}"
    _check_align(df_tr_doc, F_tr, CFG["DOC_ID_COL"])
    _check_align(df_va_doc, F_va, CFG["DOC_ID_COL"])
    _check_align(df_te_doc, F_te, CFG["DOC_ID_COL"])

    results = {}

    # ---- 1) BERT (text-only) — run BOTH MT & ST in this single run
    if args.do_doc_bert:
        print("\n[DOC] BERT (text-only)…")
        results["BERT_text_mt"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=None, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_textonly_{args.bert_mode}"),
            use_predfa=False, multi_task=True, num_workers=CFG["num_workers"]
        )
        results["BERT_text_st"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=None, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_textonly_{args.bert_mode}"),
            use_predfa=False, multi_task=False, num_workers=CFG["num_workers"]
        )

    if torch.cuda.is_available():
        try: torch.cuda.empty_cache()
        except Exception: pass

    # ---- 2) PredFA-only (MLP) — BOTH MT & ST
    if args.do_predfa_mlp:
        print("\n[DOC] PredFA-only (MLP) — HSPT-13…")
        results["PredFA_only_mt"] = run_predfa_mlp(
            df_tr_doc, df_va_doc, df_te_doc, F_tr.copy(), F_va.copy(), F_te.copy(),
            CFG["TARGETS"], CFG, outdir=os.path.join(CFG["output_dir"], "predfa_mlp_hspt13"), multi_task=True
        )
        results["PredFA_only_st"] = run_predfa_mlp(
            df_tr_doc, df_va_doc, df_te_doc, F_tr.copy(), F_va.copy(), F_te.copy(),
            CFG["TARGETS"], CFG, outdir=os.path.join(CFG["output_dir"], "predfa_mlp_hspt13"), multi_task=False
        )

    # ---- 3) BERT + PredFA (late fusion, concat) — BOTH MT & ST
    if args.use_predfa_fusion:
        print("\n[DOC] BERT + PredFA (late fusion, HSPT-13)…")
        fusion_df = predfa_agg_all
        results["BERT_plus_PredFA_mt"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=fusion_df, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_text_predfa_hspt13_{args.bert_mode}"),
            use_predfa=True, multi_task=True, num_workers=CFG["num_workers"]
        )
        results["BERT_plus_PredFA_st"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=fusion_df, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_text_predfa_hspt13_{args.bert_mode}"),
            use_predfa=True, multi_task=False, num_workers=CFG["num_workers"]
        )

    # ---- Save comparison table + pretty print with per-target Spearman + MAE
    def rows_for(tag, r):
        return {
            f"{tag}_Directness_MAE": r["MAE"][0],
            f"{tag}_Markers_MAE":   r["MAE"][1],
            f"{tag}_Overall_MAE":   r["MAE"][2],
            f"{tag}_rho_D":         r["rhos"][0],
            f"{tag}_rho_M":         r["rhos"][1],
            f"{tag}_rho_O":         r["rhos"][2],
            f"{tag}_rho_macro":     r["rho_macro"]
        }

    compare = {}
    for k,v in results.items():
        compare.update(rows_for(k, v))

    ensure_dir(CFG["output_dir"])
    with open(os.path.join(CFG["output_dir"], "compare_table.json"), "w") as f:
        json.dump(compare, f, indent=2)

    def pr_line(tag, r):
        mae = r["MAE"]; rhos = r["rhos"]
        print(f"{tag:20s} | MAE ↓  D:{mae[0]:.4f} M:{mae[1]:.4f} O:{mae[2]:.4f} | ρ_D:{rhos[0]:.4f} ρ_M:{rhos[1]:.4f} ρ_O:{rhos[2]:.4f} (macro {r['rho_macro']:.4f})")

    if results:
        print("\n=== TEST Comparison (lower MAE better, higher ρ better) ===")
        for k in sorted(results.keys()):
            pr_line(k, results[k])
    print("\n[DONE] Single run produced BOTH single-task and multi-task results for all enabled variants (HSPT-13 features).")

if __name__ == "__main__":
    main()
